In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/recovered_K.csv") 
#the dots help go from the notebook file to the airbnb file (the main file) and back into the data file

print(len(df.columns))

73


In [2]:
data = df.copy()

In [3]:
drop_cols = [
    "listing_id",
    "listing_name",
    "cover_photo_url",
    "host_id",
    "host_name",
    "cohost_ids",
    "cohost_names",
    "currency",
    "instant_book",
    "description",
    "checkin_time",
    "checkout_time",
    "photo_urls"
]

data = data.drop(columns = drop_cols, errors = "ignore")

# "Check-in and check-out times showed no meaningful variation 
#  in occupancy across groups — excluded from modeling"

print(data.dtypes[data.dtypes == 'object']) #object = str

listing_type               object
room_type                  object
amenities                  object
professional_management    object
cancellation_policy        object
guest_favorite             object
exact_location             object
single_fee_structure       object
dtype: object


In [4]:
#leakage columns to be dropped as the target column is occupany!

leakage_cols = [
    "ttm_revenue", "ttm_revenue_native", "ttm_avg_rate", "ttm_avg_rate_native",
    "ttm_adjusted_occupancy", "ttm_revpar", "ttm_revpar_native",
    "ttm_adjusted_revpar", "ttm_adjusted_revpar_native",
    "ttm_reserved_days", "ttm_blocked_days", "ttm_available_days", "ttm_total_days",
    "l90d_revenue", "l90d_revenue_native", "l90d_avg_rate", "l90d_avg_rate_native",
    "l90d_occupancy", "l90d_adjusted_occupancy", "l90d_revpar", "l90d_revpar_native",
    "l90d_adjusted_revpar", "l90d_adjusted_revpar_native",
    "l90d_reserved_days", "l90d_blocked_days", "l90d_available_days", "l90d_total_days"
]

data = data.drop(columns = leakage_cols, errors = "ignore")
#another way to write above data.drop(leakage_cols, axis = 1)


In [5]:
#clean all the data!

data["superhost"] = data["superhost"].fillna(False).astype(int)
# data["instant_book"] = data["instant_book"].fillna(False).astype(int) -> drop
data["professional_management"] = data["professional_management"].fillna(False).astype(int)
data["registration"] = data["registration"].fillna(False).astype(int)
data["guest_favorite"] = data["guest_favorite"].fillna(False).astype(int)
data["exact_location"] = data["exact_location"].fillna(False).astype(int)
data["single_fee_structure"] = data["single_fee_structure"].fillna(False).astype(int)


data["photos_count"] = data["photos_count"].fillna(data["photos_count"].median()).astype(int)
data["guests"] = data["guests"].fillna(data["guests"].median()).astype(int)
data["bedrooms"] = data["bedrooms"].fillna(data["bedrooms"].median()).astype(float)
data["beds"] = data["beds"].fillna(data["beds"].median()).astype(float)
data["baths"] = data["baths"].fillna(data["baths"].median()).astype(float)
data["min_nights"] = data["min_nights"].fillna(data["min_nights"].median()).astype(int)


data["cleaning_fee"] = data["cleaning_fee"].fillna(data["cleaning_fee"].median()).astype(float)
data["extra_guest_fee"] = data["extra_guest_fee"].fillna(0).astype(float)


rating_cols = [
    "rating_overall",
    "rating_accuracy",
    "rating_checkin",
    "rating_cleanliness",
    "rating_communication",
    "rating_location",
    "rating_value"
]

for i in rating_cols:
    data[i] = data[i].fillna(data[i].mean()).astype(float)
    

data["num_reviews"] = data["num_reviews"].fillna(0).astype(int)


data["latitude"] = data["latitude"].astype(float)
data["longitude"] = data["longitude"].astype(float)


data["room_type"] = data["room_type"].fillna(data["room_type"].mode()[0])
data["listing_type"] = data["listing_type"].fillna(data["listing_type"].mode()[0])
data["cancellation_policy"] = data["cancellation_policy"].fillna(data["cancellation_policy"].mode()[0])

data["amenities"] = data["amenities"].astype(str)
def count_amenities(x):
    return len(x.split(","))
data["amenities"] = data["amenities"].apply(count_amenities)


#drop_first is a parameter of pd.get_dummies so its reserved within that function
#one-hot encoding just means turning a categorical column with N categories into N-1 binary (0/1) columns, 
#dropping one as the baseline. Each column is just "is this category or not"

/var/folders/6r/167hkxb529jctmtgmmp2dq3w0000gn/T/ipykernel_71298/1975454121.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["professional_management"] = data["professional_management"].fillna(False).astype(int)
/var/folders/6r/167hkxb529jctmtgmmp2dq3w0000gn/T/ipykernel_71298/1975454121.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["guest_favorite"] = data["guest_favorite"].fillna(False).astype(int)
/var/folders/6r/167hkxb529jctmtgmmp2dq3w0000gn/T/ipykernel_71298/1975454121.py:8: FutureWarning: Downcasting object dtype arrays on .filln

In [6]:
#one hot coding to avoid redundancy that confuses the computer
data = pd.get_dummies(
    data,
    columns=["room_type", "listing_type", "cancellation_policy"],
    drop_first=True 
)

In [7]:
data = data.sample(frac=1, random_state=42).reset_index(drop=True) #shuffle rows before splitting so train/test are random
train_size = int(len(data) * 0.8) #240 for training 60 for testing

train = data.iloc[:train_size]  
test = data.iloc[train_size:]   

X_train = train.drop(columns=["ttm_occupancy"]).astype(float)
y_train = train["ttm_occupancy"].astype(float)

X_test = test.drop(columns=["ttm_occupancy"]).astype(float)
y_test = test["ttm_occupancy"].astype(float)

# keeping as DataFrame (not converting to numpy)
# astype(float) needed because bool columns (one-hot encoded) cause statsmodels to error
# scaler.fit_transform() will convert to numpy automatically

# .sample() = do random sampling / random order
# frac=1 = keep all rows
# random_state=42 = keep the same shuffle every time

In [8]:
#standardizing. We want to minimize errors and we dont want to give more relevance to bigger scales. 

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) #learns the mean and std then scales train, similar to applying a z formula, removes bias cause we are centering around 0 when x - mean
X_test_scaled = scaler.transform(X_test) #uses the same mean and std of the train cause we cant usually know the real data means and std

#fit() learns the mean and std
#transform() applies scalling using whatever mean and std it just learned
#.fit_transform() does both


In [9]:
#LASSO - LOOK OVER AGAIN
#higher alpha = more penalty 

from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectFromModel

#model
lassoCV = LassoCV(cv = 4, random_state = 42).fit(X_train_scaled, y_train)


print(f"Optimal Alpha: {lassoCV.alpha_}")

#repeat 4 times so each fold gets a turn as validation

Optimal Alpha: 0.00844593697198956


In [10]:
#Identify and transform data to keep only selected features 
selector = SelectFromModel(lassoCV, prefit = True)

original_feature_names = X_train.columns
selected_features = original_feature_names[selector.get_support()] #find which survived

#removes the dropped columns
X_train_scaled = selector.transform(X_train_scaled) 
X_test_scaled = selector.transform(X_test_scaled) 

print(f"Selected {len(selected_features)} features.")
print(f"Features kept: {list(selected_features)}")

#selector.get_support() -> gives a T or F, which features survived and which didn't

Selected 14 features.
Features kept: ['photos_count', 'superhost', 'professional_management', 'guest_favorite', 'single_fee_structure', 'num_reviews', 'rating_checkin', 'rating_cleanliness', 'rating_location', 'l90d_avg_length_of_stay', 'listing_type_Entire rental unit', 'listing_type_Private room in condo', 'listing_type_Private room in townhouse', 'cancellation_policy_Moderate']


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# no need to add column of 1s, sklearn handles intercept automatically
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("R²:", r2_score(y_test, y_pred)) #y_test gives you both y values and mean
print("MSE:", mean_squared_error(y_test, y_pred))

R²: 0.5844768638125922
MSE: 0.028425972258085484


In [12]:
#non-linear track

from boruta import BorutaPy
from sklearn.ensemble import RandomForestRegressor

#RF model for Boruta
rf = RandomForestRegressor(n_jobs = -1, max_depth = 5, random_state = 45)

feature_selector = BorutaPy(rf, n_estimators = 'auto', verbose = 0, random_state=42)

feature_selector.fit(X_train.values, y_train.values) #.values to convert from DF to numpy


BorutaPy(estimator=RandomForestRegressor(max_depth=5, n_estimators=74,
                                         n_jobs=-1,
                                         random_state=RandomState(MT19937) at 0x13A3C8A40),
         n_estimators='auto', random_state=RandomState(MT19937) at 0x13A3C8A40)

In [13]:
#Get the list of True/False for each column, bool
selected = feature_selector.support_

#Apply that mask to your column names to get the names
final_features = X_train.columns[selected].tolist()


print(final_features)

['amenities', 'single_fee_structure', 'num_reviews', 'l90d_avg_length_of_stay']


In [14]:
from sklearn.model_selection import RepeatedKFold, cross_val_score

#Use the subset of features found by Boruta
X_train_selected = X_train[final_features]
X_test_selected = X_test[final_features]

#Setup repeated validation (5 folds, repeated 3 times)
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

#final RF model
final_rf = RandomForestRegressor(n_estimators=100, max_depth=3, min_samples_leaf=10, max_features='sqrt',random_state=42)
final_rf.fit(X_train_selected, y_train)

#predict on untouched test set
y_pred = final_rf.predict(X_test_selected)

scores = cross_val_score(final_rf, X_train_selected, y_train, cv=rkf, scoring='r2') #r^2 for each fold -> 5 x 3 = 15

from sklearn.metrics import r2_score
print(f"Test R²:  {r2_score(y_test, y_pred)}")
print(f"CV Avg R²: {scores.mean()}")

Test R²:  0.579900364738968
CV Avg R²: 0.577783080179644


In [15]:
comparison = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print(comparison.head())

     Actual  Predicted
240   0.195   0.562230
241   0.041   0.094151
242   0.211   0.528655
243   0.055   0.285754
244   0.323   0.438390


In [16]:
data.to_csv("K_cleaned.csv", index=False)